# Inventory Data

This notebook performs comprehensive data quality checks on the Inventory dataset.


## 1. Data Loading


In [1]:
import pandas as pd

# Load the Inventory_Data data
df = pd.read_csv('../datasets/Inventory_Data.csv')

print("Data loaded successfully!")
print(f"Total records: {len(df)}")


Data loaded successfully!
Total records: 100


## 2. Basic Data Exploration


In [2]:
# Display first few rows
df.head()

,Site ID,Product ID,Beginning Inventory,Ending Inventory,Replenishment,Stockout Flag
0,A11D,PRD10087,222,233,42,No
1,AM14,PRD10082,428,442,72,No
2,ZYTJ,PRD10093,457,502,55,No
3,0T0P,PRD10092,329,378,72,No
4,BVM2,PRD10082,80,153,77,No


In [3]:
# Display column names
df.columns

Index(['Site ID', 'Product ID', 'Beginning Inventory', 'Ending Inventory',
       'Replenishment', 'Stockout Flag'],
      dtype='object')

In [4]:
# Display dataset shape
df.shape

(100, 6)

In [5]:
# Display data types and basic info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Site ID              100 non-null    object
 1   Product ID           100 non-null    object
 2   Beginning Inventory  100 non-null    int64 
 3   Ending Inventory     100 non-null    int64 
 4   Replenishment        100 non-null    int64 
 5   Stockout Flag        100 non-null    object
dtypes: int64(3), object(3)
memory usage: 4.8+ KB


## 3. Data Quality Checks

### 3.1 Missing Values Analysis


In [6]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

print("=== Missing Values Analysis ===")
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

=== Missing Values Analysis ===
                     Missing Values  Percentage
Site ID                           0         0.0
Product ID                        0         0.0
Beginning Inventory               0         0.0
Ending Inventory                  0         0.0
Replenishment                     0         0.0
Stockout Flag                     0         0.0

Total missing values: 0


### 3.2 Duplicate Records Check


In [7]:
# Check for duplicate records
duplicates_all = df.duplicated().sum()
print("=== Duplicate Records Check ===")
print(f"Total duplicate rows: {duplicates_all}")

=== Duplicate Records Check ===
Total duplicate rows: 0


### 3.3 Data Type Validation


In [8]:
# Check data types
print("=== Data Type Validation ===")
print(df.dtypes)

# Verify expected data types
expected_types = {
    'Site ID': 'object',
    'Product ID': 'object',
    'Beginning Inventory': 'int64',
    'Ending Inventory': 'int64',
    'Replenishment': 'int64',
    'Stockout Flag': 'object'
}

type_issues = []
for col, expected in expected_types.items():
    if col in df.columns and df[col].dtype != expected:
        type_issues.append(f"{col}: expected {expected}, got {df[col].dtype}")

if type_issues:
    print("\nData Type Issues:")
    for issue in type_issues:
        print(f"  - {issue}")
else:
    print("\nAll data types are correct!")

=== Data Type Validation ===
Site ID                object
Product ID             object
Beginning Inventory     int64
Ending Inventory        int64
Replenishment           int64
Stockout Flag          object
dtype: object

All data types are correct!


### 3.4 Value Range Checks


In [9]:
# Check value ranges for numeric columns
print("=== Value Range Checks ===")

# Get numeric columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    print(f"\n{col}: Min={min_val}, Max={max_val}")
    
    # Check for negative values (except where appropriate)
    if 'ID' not in col and 'Flag' not in col:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  Negative values found: {neg_count}")


=== Value Range Checks ===

Beginning Inventory: Min=53, Max=495

Ending Inventory: Min=77, Max=565

Replenishment: Min=10, Max=100


### 3.5 Categorical Value Validation


In [10]:
# Check unique values in categorical columns
print("=== Categorical Value Validation ===")

cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if 'ID' not in col and 'Date' not in col and 'Name' not in col:
        unique_vals = df[col].unique()
        print(f"\n{col} unique values: {unique_vals}")


=== Categorical Value Validation ===

Stockout Flag unique values: ['No']


### 3.6 Outlier Detection


In [11]:
# Detect outliers using IQR method
print("=== Outlier Detection (IQR Method) ===")

def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# Check outliers for numeric columns
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_columns:
    if 'ID' not in col:
        count, lower, upper = detect_outliers_iqr(col)
        print(f"\n{col}:")
        print(f"  Outliers: {count}")
        print(f"  Lower Bound: {lower:.2f}")
        print(f"  Upper Bound: {upper:.2f}")


=== Outlier Detection (IQR Method) ===

Beginning Inventory:
  Outliers: 0
  Lower Bound: -179.00
  Upper Bound: 697.00

Ending Inventory:
  Outliers: 0
  Lower Bound: -126.75
  Upper Bound: 695.25

Replenishment:
  Outliers: 0
  Lower Bound: -22.88
  Upper Bound: 148.12


## 4. Data Quality Summary


In [12]:
# Generate comprehensive data quality summary
print("=" * 50)
print("DATA QUALITY SUMMARY")
print("=" * 50)

print(f"\n1. Dataset Overview:")
print(f"   - Total Records: {len(df)}")
print(f"   - Total Columns: {len(df.columns)}")

print(f"\n2. Missing Values:")
print(f"   - Total Missing: {df.isnull().sum().sum()}")
for col in df.columns:
    missing = df[col].isnull().sum()
    print(f"   - {col}: {missing}")

print(f"\n3. Duplicate Records:")
print(f"   - Duplicate Rows: {df.duplicated().sum()}")

print(f"\n4. Data Quality Status:")
quality_passed = True

if df.isnull().sum().sum() > 0:
    print(f"   ❌ Missing values found")
    quality_passed = False
else:
    print(f"   ✓ No missing values")

if df.duplicated().sum() > 0:
    print(f"   ❌ Duplicate records found")
    quality_passed = False
else:
    print(f"   ✓ No duplicate records")

if quality_passed:
    print(f"\n✅ DATA QUALITY CHECK PASSED!")
else:
    print(f"\n❌ DATA QUALITY ISSUES DETECTED - REVIEW REQUIRED!")


DATA QUALITY SUMMARY

1. Dataset Overview:
   - Total Records: 100
   - Total Columns: 6

2. Missing Values:
   - Total Missing: 0
   - Site ID: 0
   - Product ID: 0
   - Beginning Inventory: 0
   - Ending Inventory: 0
   - Replenishment: 0
   - Stockout Flag: 0

3. Duplicate Records:
   - Duplicate Rows: 0

4. Data Quality Status:
   ✓ No missing values
   ✓ No duplicate records

✅ DATA QUALITY CHECK PASSED!
